In [1]:
import numpy as np
import pandas as pd
import gc
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
import datetime
from datetime import timedelta
import random
from imblearn.over_sampling import RandomOverSampler
from numba import njit
import numba as nb
from scipy.stats import zscore
import json
from scipy.stats import spearmanr
import websocket
import warnings
warnings.filterwarnings('ignore')
import random as python_random
import statsmodels.api as sm
import joblib
import sys
import time
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F 
import os
import importlib
TL = importlib.import_module("torch")
NN = TL.nn
F = importlib.import_module("torch.nn.functional")

from scipy.stats import spearmanr
np.random.seed(150)
python_random.seed(150)
import itertools
import copy

In [8]:
hold_pos = 20

In [9]:
def scan_dataframe_fast_dynamic(df, periods=90, q_lookback=180):
    log_prices_arr = df['Log'].to_numpy()
    

    # Executa a função dinâmica (Q e R mudam a cada linha do dataframe)
    sum_nis_array = run_rolling_nis_dynamic_numba(
        log_prices_arr, periods, q_lookback, Q_scale=0.01, R_scale=10.0
    )

    # Wilson-Hilferty para Chi-Squared 95%
    df_degrees = periods - 1
    z_90 = 1.2815515655446004
    z_95 = 1.6448536269514722
    z_99 = 2.326
    critical_value = df_degrees * (1.0 - 2.0 / (9.0 * df_degrees) + z_95 * np.sqrt(2.0 / (9.0 * df_degrees))) ** 3

    df['sum_nis'] = sum_nis_array
    df['is_stationary'] = sum_nis_array <= critical_value
    df['market_regime'] = np.where(
        df['sum_nis'].isna(), 'WARMUP',
        np.where(df['is_stationary'], 'STATIONARY', 'NON-STATIONARY')
    )
    
    return df

In [10]:
class SwiGLULayer(NN.Module):
    def __init__(self, in_f, out_f, drop):
        super().__init__()
        self.dense = NN.Linear(in_f, out_f*2, bias=False)
        self.drop = NN.Dropout(drop)
    def forward(self, x):
        g, v = TL.chunk(self.dense(x), 2, dim=-1)
        return self.drop(F.silu(g) * v)
class SwiGLUBaseModel(NN.Module):
    def __init__(self, in_dim, hid, drop):
        super().__init__()
        self.ln_in = NN.LayerNorm(in_dim)
        self.l1 = SwiGLULayer(in_dim, hid, drop)
        self.ln1 = NN.LayerNorm(hid)
        self.l2 = SwiGLULayer(hid, hid, drop)
        self.ln2 = NN.LayerNorm(hid)
        self.out = NN.Linear(hid, 1)
    def forward(self, x):
        x = self.ln_in(x)
        x = self.l1(x)
        x = self.ln1(x)
        x = self.l2(x)
        x = self.ln2(x)
        return self.out(x)

In [11]:
# Load your 0.1910 model - 10 features, 256 hidden
device = TL.device('cuda' if TL.cuda.is_available() else 'cpu')
model = SwiGLUBaseModel(11, 256, 0.0).to(device)  # 10 features, 256 hidden - from your last run
model.load_state_dict(TL.load('best_11f_256_1868.pt', map_location=device))
model.eval()

SwiGLUBaseModel(
  (ln_in): LayerNorm((11,), eps=1e-05, elementwise_affine=True, bias=True)
  (l1): SwiGLULayer(
    (dense): Linear(in_features=11, out_features=512, bias=False)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (l2): SwiGLULayer(
    (dense): Linear(in_features=256, out_features=512, bias=False)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (out): Linear(in_features=256, out_features=1, bias=True)
)

In [12]:
@njit
def fast_poc_expanding_numba(close, volume, max_window=300, min_periods=70, bins=20):
    n = len(close)
    out = np.full(n, np.nan)

    # Começamos a calcular apenas após atingir o min_periods (70)
    for i in range(min_periods - 1, n):
        # Lógica da Janela Expansiva:
        # Se i < 300, a janela é [0 : i+1] (tamanho i+1)
        # Se i >= 300, a janela é [i-299 : i+1] (tamanho fixo 300)
        start = max(0, i - max_window + 1)

        w_close = close[start : i + 1]
        w_vol = volume[start : i + 1]

        p_min = np.min(w_close)
        p_max = np.max(w_close)

        if p_min >= p_max:
            out[i] = 0.0
            continue

        # Histograma manual (Otimizado para Numba)
        hist = np.zeros(bins)
        bin_width = (p_max - p_min) / bins

        for j in range(len(w_close)):
            # Calcula o bin correspondente
            val = w_close[j]
            idx = int((val - p_min) / bin_width)
            if idx >= bins: 
                idx = bins - 1 # Garante que o valor máximo entre no último bin
            hist[idx] += w_vol[j]

        # Encontra o bin com maior volume (Point of Control)
        max_bin_idx = np.argmax(hist)
        poc = p_min + (max_bin_idx + 0.5) * bin_width

        # p_last / poc - 1 (Sinal de desvio do preço médio de volume)
        out[i] = close[i] / poc - 1

    return out

# call function
def VPRO2(df,  max_window=300, min_periods=70, bins=20):
    df['Vpro_20'] = fast_poc_expanding_numba(
        df['close'].values, 
        df['volume'].values, 
        max_window=300, 
        min_periods=70,
        bins=20
    )
    return df

############### AARON #######################################
def ARO2(df):

    @njit
    def fast_aroon_osc(high, low, window):
        n = len(high)
        out = np.full(n, np.nan)

        for i in range(window - 1, n):
            max_idx = 0
            min_idx = 0
            max_val = high[i - window + 1]
            min_val = low[i - window + 1]

            for j in range(1, window):
                curr_h = high[i - window + 1 + j]
                curr_l = low[i - window + 1 + j]

                if curr_h >= max_val:
                    max_val = curr_h
                    max_idx = j
                if curr_l <= min_val:
                    min_val = curr_l
                    min_idx = j

            days_since_max = (window - 1) - max_idx
            days_since_min = (window - 1) - min_idx

            up = ((window - days_since_max) / window) * 100
            down = ((window - days_since_min) / window) * 100
            
            out[i] = up - down
            
        return out

    # Direct extraction and single function execution for window = 15
    df['AAR_195'] = fast_aroon_osc(df['high'].values, df['low'].values, 195)
    
    return df
##################################################################

###### Weighted MACD short term ######


@nb.njit(cache=True)
def ewm_numba(arr, span):
    """Calculates Exponential Moving Average matching Pandas 'adjust=False'."""
    n = len(arr)
    out = np.empty(n, dtype=np.float64) # 🟢 FIXED: Allocates a clean, isolated array in memory
    if n == 0:
        return out

    alpha = 2.0 / (span + 1.0)

    # Safely prime the first valid element
    if n > 0:
        out[0] = arr[0]

    for i in range(1, n):
        if np.isnan(arr[i]):
            out[i] = out[i-1]
        elif np.isnan(out[i-1]):
            out[i] = arr[i]
        else:
            out[i] = arr[i] * alpha + out[i-1] * (1.0 - alpha)
    return out

@nb.njit(cache=True)
def rolling_mean_numba(arr, window):
    """Calculates standard Simple Moving Average (SMA)."""
    n = len(arr)
    out = np.empty(n)
    out[:] = np.nan

    if n < window:
        return out

    current_sum = 0.0
    for i in range(window):
        current_sum += arr[i]
    out[window - 1] = current_sum / window

    for i in range(window, n):
        current_sum += arr[i] - arr[i - window]
        out[i] = current_sum / window
    return out

@nb.njit(parallel=True, cache=True)
def process_signals_numba(close,n1, n2, n3):
    """Processes MACD EWM/Rolling layers and handles vectorized signal logic."""
    n = len(close)

    # --- 1. EXPONENTIAL MACD LAYER (EWM) ---
    ma_fast_ewm = ewm_numba(close, n1)
    ma_slow_ewm = ewm_numba(close, n2)
    ma_ewm = ma_fast_ewm - ma_slow_ewm
    sig_ewm = ewm_numba(ma_ewm, n3)
    macds_ewm = ma_ewm - sig_ewm
    macd_out = np.zeros(n)

    # --- 2. ROLLING MACD LAYER (SMA) ---
    ma_fast_roll = rolling_mean_numba(close, n1)
    ma_slow_roll = rolling_mean_numba(close, n2)
    ma_roll = ma_fast_roll - ma_slow_roll
    sig_roll = ewm_numba(ma_roll, n3)
    macds_roll = ma_roll - sig_roll
    macdn_out = np.zeros(n)

    
    return macd_out, ma_ewm

# --- 2. MAIN INTEGRATION WRAPPER FOR PANDAS DATAFRAMES ---

def execute_strategy_numba(df):
    """Extracts NumPy arrays from DataFrame, triggers Numba, and commits back."""
    close = np.ascontiguousarray(df['close'].to_numpy(dtype=np.float64))
    low = np.ascontiguousarray(df['low'].to_numpy(dtype=np.float64))
    high = np.ascontiguousarray(df['high'].to_numpy(dtype=np.float64))
    volume = np.ascontiguousarray(df['volume'].to_numpy(dtype=np.float64))


    # Step 2:  execution
    i=84
    n1, n2, n3 = i, 2*i, int(i/1.5) # 🟢 OPTIMIZED: Forces explicit int signature to ensure clean compilation x, 2x, x/1.5

    macd_out, ma_ewm = process_signals_numba(
        close, n1, n2, n3
    )

    #df[f'MACD_{i}'] = macd_out
    df[f'MA_{i}'] = ma_ewm
    #df[f'MACDS_{i}'] = macds_ewm
    #df[f'MACDN_{i}'] = macdn_out

    return df

def WMA2(df):
    # Execute the strategy using your DataFrame
    df = execute_strategy_numba(df)
    return df

###################### VWAP ############################################

#@njit
def fast_vwap_zscore(close, high, low, volume, max_window=300, min_periods=30):
    n = len(close)
    vwap_z = np.full(n, np.nan)

    # Preço Típico ponderado por Volume: (H+L+C)/3 * Vol
    tp_vol = ((close + high + low) / 3.0) * volume

    for i in range(min_periods - 1, n):
        # Define o início da janela (expansiva até max_window)
        start = max(0, i - max_window + 1)

        # Somas para o VWAP
        sum_tp_vol = np.sum(tp_vol[start : i + 1])
        sum_vol = np.sum(volume[start : i + 1])

        if sum_vol == 0:
            vwap_z[i] = 0.0
            continue

        current_vwap = sum_tp_vol / sum_vol

        # Para o Z-Score (df['Close'] - vwap) / std(vwap)
        # Note: Seu código original calcula o desvio padrão do HISTÓRICO do VWAP
        if i >= start + min_periods:
            # Precisamos calcular o desvio padrão dos valores de VWAP na janela
            # Para performance, vamos extrair o histórico de VWAPs parciais
            window_vwaps = np.zeros(i - start + 1)
            temp_tp_vol = 0.0
            temp_vol = 0.0

            # Este sub-loop calcula o histórico de VWAPs para o desvio padrão
            for k in range(start, i + 1):
                temp_tp_vol = np.sum(tp_vol[start : k + 1])
                temp_vol = np.sum(volume[start : k + 1])
                window_vwaps[k - start] = temp_tp_vol / temp_vol if temp_vol != 0 else current_vwap

            std_vwap = np.std(window_vwaps)

            if std_vwap != 0:
                vwap_z[i] = (close[i] - current_vwap) / std_vwap
            else:
                vwap_z[i] = 0.0

    return vwap_z

def VWAP2(df):
    df['VWAP_300'] = fast_vwap_zscore(
        df['close'].values, df['high'].values, 
        df['low'].values, df['volume'].values, 300
    )
    return df
    
########################## CCI #############################
    
def CCI2(df):

    #@njit
    def calcular_cci_fast(typ_price, window):
        n = len(typ_price)
        out = np.full(n, np.nan)

        for i in range(window - 1, n):
            # 1. Select window Tipical Price
            janela = typ_price[i - window + 1 : i + 1]

            # 2. MA Tipical price (TP_MA)
            tp_ma = np.mean(janela)

            # 3.Absolute Deviation (MAD) 
            # 
            mad = np.mean(np.abs(janela - tp_ma))

            # 4. Cálculate do CCI
            if mad != 0:
                out[i] = (typ_price[i] - tp_ma) / (0.015 * mad)
            else:
                out[i] = 0

        return out

    # Integration on the Loop
    tp = ((df['high'] + df['low'] + df['close']) / 3).values

    i = 150
    df[f'CCI_{i}'] = calcular_cci_fast(tp, i)

    return df

############### RSI ##########################

def RSI2(df):
    df['S1_pct'] = df['close'].pct_change()

    #@njit
    def calcular_rsi_wilder(changes, window):
        n = len(changes)
        rsi = np.full(n, np.nan)

        # Separate gains and losses (Vetorization Numba)
        gains = np.where(changes > 0, changes, 0.0)
        losses = np.where(changes < 0, -changes, 0.0)

        # Inicialization (Simple avg for the first value)
        avg_gain = np.mean(gains[:window])
        avg_loss = np.mean(losses[:window])

        if avg_loss == 0:
            rsi[window-1] = 100
        else:
            rs = avg_gain / (avg_loss + 1e-10)
            rsi[window-1] = 100 - (100 / (1 + rs))

        # Wilder softness (Recursive)
        for i in range(window, n):
            avg_gain = (avg_gain * (window - 1) + gains[i]) / window
            avg_loss = (avg_loss * (window - 1) + losses[i]) / window

            if avg_loss == 0:
                rsi[i] = 100
            else:
                rs = avg_gain / (avg_loss + 1e-10)
                rsi[i] = 100 - (100 / (1 + rs))

        return rsi - 50 # Retorna centralizado no 0 conforme seu código

    # Loop integration
    # S1_pct pct change of Close
    changes = df['S1_pct'].values

    i = 160
    df[f'RSI_{i}'] = calcular_rsi_wilder(changes, i)
        
    return df
        
    

########################## HA ####################################

def HA2(df):

    #@njit # Just-In-Time compilation for max performance 
    def calcular_ha_core(op, hi, lo, cl):
        n = len(op)
        ha_open = np.empty(n)
        ha_close = (op + hi + lo + cl) / 4.0

        # First candle (Seed)
        ha_open[0] = (op[0] + cl[0]) / 2.0

        # Loop recursive 
        for i in range(1, n):
            ha_open[i] = (ha_open[i-1] + ha_close[i-1]) / 2.0

        # High and low
        ha_high = np.maximum(hi, np.maximum(ha_open, ha_close))
        ha_low = np.minimum(lo, np.minimum(ha_open, ha_close))

        # Vetorization of signals (1 for up, -1 for down, 0 neutral)
        sinal = np.zeros(n)
        for i in range(n):
            if (ha_close[i] > ha_open[i]) and (ha_open[i]==ha_low[i]):
                sinal[i] = 1.0
            elif (ha_close[i] > ha_open[i]) and (ha_close[i-1] < ha_open[i]):
                sinal[i] = 1.0
            
            elif (ha_close[i] < ha_open[i]) and (ha_open[i]==ha_high[i]):
                sinal[i] = -1.0  
            elif (ha_close[i] < ha_open[i]) and (ha_close[i-1] > ha_open[i]):
                sinal[i] = -1.0

        return sinal

    def sequence_counter(sinal):
        # Lógic to count sequence (Reset on 0)
        # Convert to NumPy to avoid overhead groups Pandas
        n = len(sinal)
        ha_count = np.zeros(n)
        current_sum = 0.0

        for i in range(n):
            if sinal[i] == 0:
                current_sum = 0.0
            elif i > 0 and sinal[i] != sinal[i-1]:
                current_sum = sinal[i] # Reset e começa nova contagem
            else:
                current_sum += sinal[i]
            ha_count[i] = current_sum
        return ha_count

    # Integration on DataFrame
    df['HA_Sinal'] = calcular_ha_core(df['open'].values, df['high'].values, 
                                      df['low'].values, df['close'].values)

    df['HA_Seq'] = sequence_counter(df['HA_Sinal'].values)
        
        
    return df

 ###############################  Shadow_MA  ##################################################### 
    
@njit
def calc_sma_numba(arr, period):
    n = arr.shape[0]
    out = np.full(n, np.nan, dtype=np.float64)

    if n >= period:
        current_sum = 0.0
        for i in range(period):
            current_sum += arr[i]
        out[period - 1] = current_sum / period

        for i in range(period, n):
            current_sum = current_sum - arr[i - period] + arr[i]
            out[i] = current_sum / period
    return out

# 2. FUNÇÃO PRINCIPAL: Calcula as dinâmicas brutas de candle
@njit
def calc_candle_raw_numba(open_arr, high_arr, low_arr, close_arr):
    n = open_arr.shape[0]
    upper_s = np.empty(n, dtype=np.float64)
    lower_s = np.empty(n, dtype=np.float64)
    body_tr = np.empty(n, dtype=np.float64)

    for i in range(n):
        candle_range = (high_arr[i] - low_arr[i]) + 1e-8
        max_body = open_arr[i] if open_arr[i] > close_arr[i] else close_arr[i]
        min_body = close_arr[i] if open_arr[i] > close_arr[i] else open_arr[i]

        upper_s[i] = (high_arr[i] - max_body) / candle_range
        lower_s[i] = (min_body - low_arr[i]) / candle_range
        body_tr[i] = abs(close_arr[i] - open_arr[i]) / candle_range

    return upper_s, lower_s, body_tr

def SHADOW(df):

    # --- PREPARAÇÃO DOS DADOS ---
    o = df['open'].values.astype(np.float64)
    h = df['high'].values.astype(np.float64)
    l = df['low'].values.astype(np.float64)
    c = df['close'].values.astype(np.float64)

    # Passo 1: Calcula o básico fixo uma única vez (Economiza muito processamento!)
    up_s_raw, lw_s_raw, b_tr_raw = calc_candle_raw_numba(o, h, l, c)

    # Salva as brutas de 1 minuto no DataFrame
    df['Upper_Shadow_Pct'] = up_s_raw
    df['Lower_Shadow_Pct'] = lw_s_raw
    df['Body_to_Range'] = b_tr_raw

    # --- O SEU FOR LOOP PARA DIVERSAS JANELAS ---
    # O range(15, 61, 15) vai testar as janelas: 15, 30, 45, 60
    JANELA_SUAVIZACAO1 = 150
    JANELA_SUAVIZACAO2 = 165
    # Calcula a média móvel no Numba de forma instantânea para a janela atual
    df[f'Upper_Shadow_MA{JANELA_SUAVIZACAO1}'] = calc_sma_numba(up_s_raw, JANELA_SUAVIZACAO1)
    df[f'Lower_Shadow_MA{JANELA_SUAVIZACAO2}'] = calc_sma_numba(lw_s_raw, JANELA_SUAVIZACAO2)


    #df = df.dropna()
    #df = df.reset_index(drop=True)

    return df

##################### TGT ########################

def TGT(df, hold_pos=hold_pos): 
    
    #target_price = df['Avg'].shift(-hold_pos).ewm(span=int(hold_pos), adjust=False).mean()
    #target_price = df['avg'].shift(-hold_pos).rolling(int(hold_pos/2)).mean()
    #df['Tgt'] = (target_price.values / df['avg'].values) - 1

    target_price1 = df['avg'].shift(int(-hold_pos))
    #target_price2 = df['Avg'].shift(int(-hold_pos*0.9))
    #target_price3 = df['Avg'].shift(int(-hold_pos*0.8))
    #target_price4 = df['Avg'].shift(int(-hold_pos*0.7))
    #target_price5 = df['Avg'].shift(int(-hold_pos*0.6))
    #target_price6 = df['Avg'].shift(int(-hold_pos*0.5))

    # 2. Realize a operação usando valores brutos (numpy) para economizar RAM
    df["ret1"] = (target_price1.values / df['avg'].values) - 1
    #df["ret2"] = (target_price2.values / df['Avg'].values) - 1
    #df["ret3"] = (target_price3.values / df['Avg'].values) - 1
    #df["ret4"] = (target_price4.values / df['Avg'].values) - 1
    #df["ret5"] = (target_price5.values / df['Avg'].values) - 1
    #df["ret6"] = (target_price6.values / df['Avg'].values) - 1

    #df['Tgt'] =(df['ret1'] + df['ret2'] + df['ret3'] + df['ret4'] + df['ret5'] + df['ret6'])/6
    #df['Tgt'] =(df['Tgt'] + df['ret1'] +  df['ret4'])/3
    #df['Tgt'] = df['ret1']



    # Define a historical lookback window (e.g., 500 or 1000 rows)
    #z_lookback = 200  

    # Calculate rolling statistical baselines based ONLY on past data
    #rolling_mean = df['Tgt'].rolling(window=z_lookback, min_periods=100).mean().shift(hold_pos)
    #rolling_std = df['Tgt'].rolling(window=z_lookback, min_periods=100).std().shift(hold_pos)
    #df["Mean"] = rolling_mean
    #df["Std"] = rolling_std

    # Apply the Z-score transformation: (X - Mean) / Std
    # We add a tiny epsilon (1e-8) to prevent zero-division errors if volatility crushes to 0
    #df['Tgt'] = (df['Tgt'] - rolling_mean) / (rolling_std + 1e-8)

     #inconcistency
    #df['Tgt'] = np.where((df['Tgt'] > 0) & (df['ret1'] > 0) | 
    #                     (df['Tgt'] < 0) & (df['ret1'] < 0), df['Tgt'], 0)





    df['Trig'] = df['ret1'].abs().rolling(25).mean()         
    #df['Trig'] = df['ret'].abs().rolling(5).mean() 
    #move trig ahead not to use future data
    df['Trig'] = df['Trig'].shift(hold_pos)

    return df

In [ ]:
# ==============================================================================
# STEP 1: PRE-EMPTIVE CLEANUP OF LINGERING MEMORY SOCKETS
# ==============================================================================
# If a 'ws' object exists from a previous crashed run, force-terminate it now
if 'ws' in globals():
    try:
        print("Wiping old lingering WebSocket connection from memory...")
        ws.close()
        del ws
    except Exception:
        pass

# ==============================================================================
# STEP 2: BLINDED LIFECYCLE WRAPPER
# ==============================================================================
def start_pipeline_stream():
    global ws
    try:
        print("Initializing clean WebSocket connection tunnel...")
        
        # PLACE YOUR ORIGINAL WEBSOCKET LAUNCH LINE HERE. For example:
        # ws = websocket.WebSocketApp("wss://://binance.com", ...)
        # ws.run_forever()
        
    except KeyboardInterrupt:
        print("\nPipeline execution stopped manually by user.")
    except Exception as e:
        print(f"Critical Pipeline Execution Crash: {str(e)}")
    finally:
        # The finally block GUARANTEES that even if your ATR or Keras code explodes,
        # the network tunnel is unlinked, allowing immediate cell re-runs.
        print("Executing emergency resource cleanup. Unlinking network sockets...")
        try:
            ws.close()
        except NameError:
            pass
        print("Success: System state restored. Ready for re-run without kernel restart!")

# Launch the secure stream
start_pipeline_stream()








# ==========================================
# 1. LIVE DATA STATE CONTAINER
# ==========================================
class TradingState:
    def __init__(self):
        # Raw tick stream accumulator
        # Explicitly instantiating the dynamic state container
                
        self.df_acc = pd.DataFrame(columns=['symbol', 'open', 'high', 'low', 'close', 'volume', 't'])
        self.df_acc.index.name = 'timestamp'
        
        # Predefined structural schema to prevent KeyError 'Open'
        self.df_final = pd.DataFrame(columns=['symbol', 'open', 'high', 'low', 'close', 'volume', 'avg',
                                             'my_p', 'sig1', 'sig2', 'pnl', 'fee', 'real_re', 'trade_pr',
                                              'pos_count'])
        
        self.df_final.index.name = 'timestamp'
        
        
        self.df_final_lim = pd.DataFrame()
        self.count = 0  # Global execution tick counter
        self.next_sig = "zero" 
        self.realized_re = 0      
        self.ntrades = 0  
        

# Initialize the single global state object instance
state = TradingState()

# Initialize, load your saved weights safely, and evaluate
# Load your 
device = TL.device('cuda' if TL.cuda.is_available() else 'cpu')
model = SwiGLUBaseModel(11, 256, 0.0).to(device)  # 10 features, 256 hidden - from your last run
model.load_state_dict(TL.load('best_11f_256_1868.pt', map_location=device))
model.eval()





try:
    my_production_scaler = joblib.load('my_production_scaler_ETH_reg.pkl')
    #my_production_pca = joblib.load('my_production_pca_ETH.pkl')
    print("### Production Scaler successfully loaded into memory! ###", flush=True)
except FileNotFoundError:
    print("### ERROR: 'my_production_scaler.pkl' or PCA not found! Please export it from your training environment. ###", flush=True) 




def process_pipeline_sync(source_data):
    
    global state, model
    # HYPERPARAMETERS
    
    max_time = 50
    min_time = 3
    max_pos = 3
    up = 10.6 #0.4 #0.386
    up2 = 0
    do = -10.6 #-0.4 #-0.376
    do2 = 0
    mult_up = 0.7
    mult_do = 0.635

    k = 4 #2.5
    k_stop = -0.5 #-1.5
    hold_pos = 20
    #reference price
    price1 = 2492.76
    start_window = 200
    window = 200
    l = window + 300
    
        
    try:
        payload = json.loads(source_data)
        
        # Unpack multiplexed stream packets instantly
        if 'data' in payload:
            payload = payload['data']
            
        # Early rejection of keep-alives or API metadata responses
        if 'result' in payload or ('id' in payload and 'k' not in payload):
            return

        if 'k' not in payload:
            return

        kline = payload['k']
        symbol = kline['s']
        open_p = float(kline['o'])
        high_p = float(kline['h'])
        low_p = float(kline['l'])
        close_p = float(kline['c'])
        volume = float(kline['v'])
        t_timestamp = kline['t']
        event_time = pd.to_datetime(payload.get('E', t_timestamp), unit='ms')

        # Build clean standalone tick frame slice
        tick_data = {
            'symbol': [str(symbol)], 'open': [open_p], 'high': [high_p],
            'low': [low_p], 'close': [close_p], 'volume': [volume], 't': [int(t_timestamp)]
        }
        df_tick = pd.DataFrame(tick_data, index=[event_time])
        df_tick.index.name = 'timestamp'
        
        # 1. ALWAYS feed the background streaming accumulator for temporal calculations (.diff/.shift)
        state.df_acc = pd.concat([state.df_acc, df_tick], ignore_index=False)

        # Enforce memory constraint boundaries
        if len(state.df_acc) > window:
            state.df_acc = state.df_acc.tail(window)

        # Filter operations to prioritize current asset target
        df_acc_1 = state.df_acc[state.df_acc['symbol'] == 'ETHUSDT'].copy()
        if df_acc_1.empty:
            return

        # 2. Feature engineering steps calculated on fast streaming arrays
        df_acc_1['dif'] = df_acc_1['t'].diff()
        df_acc_1['dif2'] = df_acc_1['dif'].shift(-1)
        
        if not df_acc_1.empty:
            df_acc_1.iat[0, df_acc_1.columns.get_loc('dif2')] = 0.0
            
        df_acc_1['avg'] = (df_acc_1['open'] + df_acc_1['close'] + df_acc_1['high'] + df_acc_1['low']) / 4.0
        
        # =================================================================
        # GATE FILTER: HALT PROCESSING UNTIL THE OFFICIAL 1M CANDLE CLOSES
        # =================================================================
        if not kline['x']:
            return

        state.count += 1
        candle_time = pd.to_datetime(df_acc_1.index[-1])
        
        # Clean stray non-datetime entries safely without resetting structural parameters
        if not state.df_final.empty:
            state.df_final.index = pd.to_datetime(state.df_final.index, errors='coerce')
            state.df_final = state.df_final[state.df_final.index.notna()]

        # Protect data feed from partial or missing index data during time rollover
        live_candle = df_acc_1.iloc[-1]
        if pd.isna(live_candle['open']) or pd.isna(live_candle['close']):
            data_source = df_acc_1.iloc[-2]
        else:
            data_source = live_candle

        # ==============================================================================
        # STABLE TIME-SERIES CONCATENATION (Preserves Column Primitives)
        # ==============================================================================
        candle_time_valid = pd.to_datetime(candle_time)

        # 1. Create a clean, single-row DataFrame with explicit float types
        new_candle_df = pd.DataFrame({
            'symbol': [str(data_source['symbol'])],
            'open': [float(data_source['open'])],
            'high': [float(data_source['high'])],
            'low': [float(data_source['low'])],
            'close': [float(data_source['close'])],
            'volume': [float(data_source['volume'])],
            'avg': [float(data_source['avg'])],
            'my_p': [0.0], 
            'sig1': [0.0],
            'sig2': [0.0], 'pnl': [0.0],
            'fee': [0.0], 'real_re': [0.0],
            'trade_pr': [0.0],
            'pos_count': [0.0]
        }, index=[candle_time_valid])
        new_candle_df.index.name = 'timestamp'
        
                   
            
        ####################################3    
                    
               
                
        if len(state.df_final)>=2:
            
            #Defibne the time for later in the code
            last_timestamp = state.df_final.index[-1]
            #last_timestamp = candle_time_valid
            
            if state.next_sig == 'long_1':

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]

                # 2. Grab the reference price safely using the .at accessor
                trade_price_ref = state.df_final['avg'].at[last_timestamp]

                # 3. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = 1
                state.df_final.at[last_timestamp, 'my_p'] = 1
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'short_1':

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]

                # 2. Grab the reference price safely using the .at accessor
                trade_price_ref = state.df_final['avg'].at[last_timestamp]

                # 3. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = -1
                state.df_final.at[last_timestamp, 'my_p'] = -1
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'long_2':

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]

                # 2. Grab the reference price safely using the .at accessor
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]

                # 3. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = 1
                state.df_final.at[last_timestamp, 'my_p'] = int(pos_ref + 1)
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref

            elif state.next_sig == 'close':     

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]

                # 2. Grab the reference price safely using the .at accessor
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]

                # 3. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = -pos_ref
                state.df_final.at[last_timestamp, 'my_p'] = 0
                state.df_final.at[last_timestamp, 'pos_count'] = 0
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref



            elif state.next_sig == 'hold':

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]

                # 2. Grab the reference price safely using the .at accessor
                count_ref = state.df_final['pos_count'].ffill().iloc[-2]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]
                #trade_price_ref = state.df_final['avg'].at[last_timestamp]

                # 3. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = 0
                state.df_final.at[last_timestamp, 'my_p'] = pos_ref 
                state.df_final.at[last_timestamp, 'pos_count'] = int(count_ref + 1)
                state.df_final.at[last_timestamp, 'trade_pr'] = 0 #trade_price_ref




            elif state.next_sig == 'short_2':

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]

                # 2. Grab the reference price safely using the .at accessor
                trade_price_ref = state.df_final['avg'].at[last_timestamp]
                pos_ref = state.df_final['my_p'].ffill().iloc[-2]

                # 3. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = -1
                state.df_final.at[last_timestamp, 'my_p'] = int(pos_ref - 1)
                state.df_final.at[last_timestamp, 'pos_count'] = 1
                state.df_final.at[last_timestamp, 'trade_pr'] = trade_price_ref
            
            elif state.next_sig == 'zero':

                # 1. Capture the exact timestamp of the current active row
                #last_timestamp = state.df_final.index[-1]
                
                # 2. Atomically update the columns directly in the real DataFrame memory
                state.df_final.at[last_timestamp, 'sig1'] = 0
                state.df_final.at[last_timestamp, 'my_p'] = 0
                state.df_final.at[last_timestamp, 'pos_count'] = 0
                state.df_final.at[last_timestamp, 'trade_pr'] = 0
            

       
            
        
            
            
            
            
            
            
            
            
            
        
        # 2. Before merging, forcefully drop any duplicate columns that got stuck in memory
        if not state.df_final.empty:
            state.df_final = state.df_final.loc[:, ~state.df_final.columns.duplicated()].copy()

        # 3. Concatenate using explicit column filtering to force absolute alignment
        state.df_final = pd.concat([state.df_final, new_candle_df], axis=0, join='outer')

        # 4. HARD-DROP RECOVERY: Instantly purge any row that collapsed into NaN
        state.df_final = state.df_final.dropna(subset=['open', 'close', 'avg'])

        # ==============================================================================
        # POST-PROCESSING & PERFECT CHRONOLOGICAL ORDER
        # ==============================================================================
        state.df_final = state.df_final[~state.df_final.index.duplicated(keep='last')]
        state.df_final = state.df_final.sort_index()



        # Strictly enforce index naming
        state.df_final.index.name = 'timestamp'

        # Memory management protection for df_final
        if len(state.df_final) >window:
            state.df_final = state.df_final.tail(window)

        print(f"\n[CONFIRMED] CANDLE SAVED AT: {candle_time_valid} | Rows: {len(state.df_final)}")
        #print(state.df_final[['symbol', 'open', 'high', 'low', 'close']].tail(5))



        # 4. FIXED REFERENCE PRICE EXTRACTION (Eliminates the float ufunc log crash)
        # We extract the scalar raw float value directly from the very first row of 'open'
        #price1 = float(state.df_final['open'].head(0))

        # 5. Calculate cumulative percentage return column safely
        state.df_final['pct_return'] = ((state.df_final['close'].astype(float) - price1) / price1) * 100

        # Most indicators (ATR, RSI, CCI) require a minimum window lookback (usually 14 or 20 periods).
        # We skip indicator calculations until we accumulate enough historical 1m candles.
        MINIMUM_CANDLES_REQUIRED = 200





        if len(state.df_final) < MINIMUM_CANDLES_REQUIRED:
            print(f"\n[WARM-UP] Accumulating history: {len(state.df_final)}/{MINIMUM_CANDLES_REQUIRED} candles collected. Skipping math indicators...", flush=True)
            return

        # =================================================================
        # ####### ADVANCED MATHEMATICAL INDICATORS (INTEGRATED)
        # =================================================================
        data = state.df_final.copy()

        try:
            # CRITICAL NUMBA FIX: Force explicit primitive numeric casting on ALL core columns
            # This destroys the 'pyobject' wrapper, converting columns to clean native float64 arrays
            numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'avg']
            for col in numeric_cols:
                data[col] = pd.to_numeric(data[col], errors='coerce').astype(float)

            #print(data)

            # Type casting to prevent internal numpy ufunc float validation errors
            #data['avg'] = (data['high'] + data['low'] + data['close']) / 3.0
            data['avg'] = data['avg'].astype(float)
            data['Log'] = np.log(data['avg'])

            data = ARO2(data)
            data = CCI2(data)
            #data = ATR2(data)
            data = WMA2(data)
            data = RSI2(data)
            data = VWAP2(data)
            data = HA2(data)
            data = TGT(data)
            data = VPRO2(data)
            data = SHADOW(data)
            
            
            #print (data[['MA_12', 'MA_24','ATR_7', 'VWAP_i', 'CCI_10', 'AAR_15', 'HA_Sinal']])
            # KALMAN filter execution mapping
            data = scan_dataframe_fast_dynamic(data, periods=10, q_lookback=60)
            data['M_R'] = data['market_regime'].apply(lambda x: 1 if x == "NON-STATIONARY" else 0)
            data['M_R_roll'] = data['M_R'].rolling(3).mean()
            #print(data[['RSI_160','MA_84','VWAP_300','CCI_150','AAR_195','Vpro_20','HA_Sinal']])
            #define reference price
            if len(data) <=100:
                    price1 = data['avg'].iloc[0]

            # CRITICAL FIX: Push the computed copy back into the global state container
            # This ensures that rolling formulas like .rolling(5) or indicators have historical data next minute
            state.df_final = data.copy()

            # Verify if the trigger column exists to prevent runtime KeyErrors
            if 'Trig' in data.columns:
                # Extract the trigger value as a pure scalar float
                trig = float(data['Trig'].iloc[-1])

                # FIX: Calculate nominal boundary values based on your exact multipliers
                # These variables are now ready to be compared with future percentage matrices
                pro_ = k * trig
                stop_ = k_stop * trig

            # =================================================================
            # ####### MEMORY OPTIMIZATION & RESOURCE MANAGEMENT
            # =================================================================
            # FIX: Maintain chronological data consistency by capping the rows to 300 
            # CRITICAL: Removed .reset_index() to preserve the datetime 'timestamp' index matrix.
            # Without the datetime index, rolling metrics and future lookups will crash.
            
            if len(data) > l:
                data = data.tail(l)
                
                
                
            # Sync the memory reduction slice back into the global state container
            state.df_final = data.copy()  

            # =================================================================
            # ####### FEATURE SELECTION VECTOR FOR ML INFERENCE (x_sig)
            # =================================================================
            # Ensure all required features exist in the columns to avoid runtime KeyErrors
            required_features = ['Upper_Shadow_Pct','Lower_Shadow_Pct','Upper_Shadow_MA150','Lower_Shadow_MA165','RSI_160',
                        'MA_84','VWAP_300','CCI_150','AAR_195','Vpro_20','HA_Sinal']

            # Dynamic safety check to verify if all selected indicators were successfully computed
            missing_features = [feat for feat in required_features if feat not in data.columns]

            if not missing_features:
                # Isolate the precise feature matrix slice for your machine learning model input
                x_sig = data[required_features].copy()


            if 'x_sig' in locals() and not x_sig.empty:
                try:
                    # =================================================================
                    # ####### PRODUCTION SCALING REUSE PIPELINE
                    # =================================================================
                    # Remove any boundary rows containing NaN values generated by indicator warm-ups
                    x_sig_clean = x_sig.dropna()


                    if not x_sig_clean.empty:
                        # REUSE: The loaded scaler applies the FIXED training mean and std instantly
                        X_sig_sc_array = my_production_scaler.transform(x_sig_clean)

                        # Reconstruct the array back into a temporary DataFrame to preserve column alignment
                        X_sig_sc = pd.DataFrame(X_sig_sc_array, columns=x_sig_clean.columns, index=x_sig_clean.index)

                        # Extract the single latest row vector as a clean NumPy 2D array (.values)
                        # This vector is 100% ready to be passed into your model.predict() function
                        latest_sample_ready = X_sig_sc.tail(1).values

                        print("\n[ML INFERENCE] Scaled feature matrix generated successfully using fixed parameters.", flush=True)
                        print(f"[ML INFERENCE] Clean Vector for Model Input:\n{latest_sample_ready}", flush=True)

                        # 2. REUSE PCA: Apply fixed principal components from training
                        #Transform() from the loaded object
                        #data_pca_array = my_production_pca.transform(X_sig_sc_array)
                        data_pca_array = X_sig_sc_array

                        # 3. RECONSTRUCT DATAFRAME: Preserve index timeline for ML tracking
                        data_pca = pd.DataFrame(
                            data_pca_array,
                            columns=['PC0', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6','PC7', 'PC8', 'P95', 'PC10'],
                            index=x_sig_clean.index
                        )

                        #########################################################################

                        #print (data_pca)


                        if 'data_pca' in locals() and not data_pca.empty:

                            try:
                                # =================================================================
                                # ####### PRODUCTION KERAS INFERENCE PIPELINE
                                # =================================================================
                                latest_pca_vector = data_pca.values[-1:]

                                #latest_pca_vector = data_pca.values[-1:]
                                print ('latest_pca_vector ', latest_pca_vector )

                                # If your SAMModel expects a 3D sequential shape (batch, timesteps, features),
                                
                                # 1. Set the model to evaluation mode (safeguards BatchNorm/Dropout behaviors)
                                model.eval()
                                # 2. Extract raw data values if input is a Pandas DataFrame or standard array
                                #if hasattr(data_pca, 'values'):
                                #    data_pca_np = data_pca.values
                                #else:
                                #    data_pca_np = np.asarray(data_pca)

                                # 3. Convert input features into a clean float32 PyTorch tensor
                                data_tensor = torch.tensor(latest_pca_vector, dtype=torch.float32)

                                # 4. Automatically match the hardware device location of the model parameters (CPU/GPU)
                                model_device = next(model.parameters()).device
                                data_tensor = data_tensor.to(model_device)

                                # 5. Execute safe forward pass without computational graph overhead
                                with torch.no_grad():
                                    predictions = model(data_tensor).cpu().numpy().flatten()    
                                
                                # Sanitize target arrays by wiping data formatting variations
                                #actual_targets = y_test_sc.values.flatten() if hasattr(y_test_sc, 'values') else np.asarray(y_test_sc).flatten()
                                predicted_scores = predictions
                                
                                
                                                                                              
                                
                                #########################################################################################
                                
                                # Execute the prediction pass strictly on the active single row vector
                                y_pred_array = predictions
                                
                                
                                
                                
                                
                                
                                
                                
                                current_signal = float(y_pred_array[0])
                                
                                print('current signal ', current_signal)


                                # Reconstruct into a tiny DataFrame using the current active candle timestamp
                                candle_time_valid = pd.to_datetime(candle_time)

                                #DIRECT SAVE: Write the prediction score directly into the permanent 'y_pred' column
                                state.df_final.loc[candle_time_valid, 'y_pred'] = current_signal
                                print(f"-> Prediction score saved in y_pred: {current_signal}")
                                print('Number of minutes for mean calculation: ',state.df_final['y_pred'].count())
                                
                                
                                if state.df_final['y_pred'].count()> start_window:
                                    mean_pred = state.df_final['y_pred'].mean()
                                    #std_pred = state.df_final['y_pred'].std()
                                    up = mean_pred + mult_up #(mult_up * std_pred)
                                    do = mean_pred - mult_do #(mult_do * std_pred)
                                    
                                if state.df_final['y_pred'].count()>window:
                                    mean_pred = state.df_final['y_pred'].tail(window).mean()
                                    #std_pred = state.df_final['y_pred'].tail(window).std()
                                    up = mean_pred + mult_up #(mult_up * std_pred)
                                    do = mean_pred - mult_do #(mult_do * std_pred)     
                                    
                                print('Current thresholds: ', do, up)
                                
                                
                                
                                
                                



                                #POSITION 

                                print('--- Strategy Metrics (Tail 5) ---')
                                # Use .tail(5) 
                                if 'my_p' in state.df_final.columns:
                                    print(state.df_final[['y_pred', 'my_p', 'sig1', 'sig2','pos_count','trade_pr']].tail(5), flush=True)

                                #Calculations
                                cl = float(data['close'].iloc[-1])
                                print('Last price recorded:', cl)

                                clav = float(data['avg'].iloc[-1])


                                #Calculate profit/loss from the last trade price
                                # Ensure the trade price history series is not empty
                                if not state.df_final['trade_pr'].empty and abs(int(state.df_final['my_p'].iloc[-1])) >= 1:


                                    # FIX 1: Use .ffill() to carry forward the original entry price from past minutes.
                                    # Reading .iloc[-1] directly would yield 0 or NaN since the new row just spawned.
                                    last_valid_entry_price = float(state.df_final['trade_pr'].ffill().iloc[-1])

                                    # Safety check to avoid division by zero crashes
                                    if last_valid_entry_price > 0:
                                        # Calculate the exact percentage return shift of the active open trade
                                        tr = (cl / last_valid_entry_price) - 1.0

                                        # Store the running return percentage into the current candle log row
                                        state.df_final.loc[last_timestamp, 'trade_return'] = float(tr)
                                        print(f"[RISK SYSTEM] Active trade return (tr): {tr * 100.0:.4f}%", flush=True)
                                    else:
                                        tr = 0.0
                                else:
                                    tr = 0.0
                                    state.df_final.loc[last_timestamp, 'trade_return'] = 0.0

                                ############# START TRADING RULES #####################

                                # 1. CHECK OPEN POSITION STATE: Equivalent to 'if my_p[-1] == 0:'
                                # Bypasses list appending arrays by tracking a single scalar engine pointer
                                # 1. FIX: Use np.isclose or mathematical tolerance to safely evaluate a float zero
                                #if state.df_final.at[last_timestamp, 'my_p'] == 0
                                if state.df_final['my_p'].iloc[-2] ==0:
                                #if np.isclose(float(state.df_final['my_p'].iloc[-1]), 0.0, atol=1e-5):

                                    # 2. EVALUATE STRATEGY CONDITIONAL BOUNDARIES (Using clean chained operators)
                                    if do < current_signal < up:

                                        state.next_sig = "zero"
                                        print('No signal')


                                    # 3. OPEN LONG POSITION GATE (If score crosses the 'up' boundary AND market is stable)

                                    elif current_signal >= up and state.df_final['M_R_roll'].iloc[-1] != 1.0:

                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "long_1"
                                        state.next_sig = "long_1"
                                        print(f"Sinal LONG gravado com sucesso no candle atual!")




                                    # 3. OPEN SHORT POSITION GATE    
                                    elif current_signal <= do and state.df_final['M_R_roll'].iloc[-1] != 1.0:


                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "short_1"
                                        state.next_sig = "short_1"
                                        print(f"Sinal SHORT gravado com sucesso no candle atual!")

                                #elif state.df_final.at[last_timestamp, 'my_p'] >= 1:
                                elif state.df_final['my_p'].iloc[-2] >= 1:

                                    # 1st condition - add to position
                                    if current_signal >= up and state.df_final['my_p'].iloc[-2] < (max_pos) and state.df_final['M_R_roll'].iloc[-1] != 1.0:


                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "long_2"
                                        state.next_sig = "long_2"
                                        print(f"Sinal LONG gravado com sucesso no candle atual!")


                                    # 2nd condition - time-based close
                                    elif state.df_final['pos_count'].iloc[-2] >= max_time and current_signal < up:


                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "close"
                                        state.next_sig = "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")


                                    # 3rd condition - signal-based close
                                    elif current_signal <= do2 and state.df_final['pos_count'].iloc[-2] >= min_time:


                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "close"
                                        state.next_sig =  "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")


                                    # 4th condition - stop/profit
                                    elif tr <= stop_ or tr >= pro_:


                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "close"
                                        state.next_sig =  "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")

                                    # Hold position
                                    else:


                                        # Efind position for 'sig2' 
                                        #last_timestamp = state.df_final.index[-1]

                                        # fill directly last row
                                        # 2. Update 'sig2' directly using the label (Immune to the iloc bug)
                                        #state.df_final.at[last_timestamp, 'sig2'] = "hold"
                                        state.next_sig = "hold"
                                        print(f"Sinal HOLD gravado com sucesso no candle atual!")



                                #elif state.df_final.at[last_timestamp, 'my_p'] <= -1:
                                elif state.df_final['my_p'].iloc[-2]<= -1:

                                    # 1st condition - add to short
                                    if current_signal <= do and state.df_final['my_p'].iloc[-2] > (-max_pos) and state.df_final['M_R_roll'].iloc[-1] != 1.0:


                                        # Efind position for 'sig2' 
                                        #col_idx = state.df_final.columns.get_loc('sig2')

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "short_2"
                                        state.next_sig = "short_2"
                                        print(f"Sinal SHORT gravado com sucesso no candle atual!")



                                    # 2nd condition - time-based close
                                    elif state.df_final['pos_count'].iloc[-2] >= max_time and current_signal > do:


                                        ## find position for 'sig2' 
                                        #col_idx = state.df_final.columns.get_loc('sig2')

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "close"
                                        state.next_sig =  "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")


                                    # 3rd condition - signal-based close
                                    elif current_signal >= up2 and state.df_final['pos_count'].iloc[-2] >= min_time:


                                        # Efind position for 'sig2' 
                                        #col_idx = state.df_final.columns.get_loc('sig2')

                                        # fill directly last rowstate.
                                        #state.df_final.at[last_timestamp, 'sig2'] = "close"
                                        state.next_sig =  "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")


                                    # 4th condition - stop/profit
                                    elif -tr <= stop_ or -tr >= pro_:


                                        # Efind position for 'sig2' 
                                        #col_idx = state.df_final.columns.get_loc('sig2')

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "close"
                                        state.next_sig =  "close"
                                        print(f"Sinal CLOSE gravado com sucesso no candle atual!")


                                    # Hold position
                                    else:


                                        # Efind position for 'sig2' 
                                        #col_idx = state.df_final.columns.get_loc('sig2')

                                        # fill directly last row
                                        #state.df_final.at[last_timestamp, 'sig2'] = "hold"
                                        state.next_sig = "hold"
                                        print(f"Sinal HOLD gravado com sucesso no candle atual!")


                                print('##### next_sig#########', state.next_sig)

                                candle_time_valid = pd.to_datetime(candle_time)        
                                print('Last minute signal:', state.df_final.loc[candle_time_valid, 'sig1'] )


                                cl_avg = state.df_final.loc[candle_time_valid, 'avg']
                                pos_price = -state.df_final.loc[candle_time_valid, 'sig1'] * cl_avg 
                                
                                #extract last my_p
                                #last_my_p = state.df_final['my_p'].iloc[-2]

                                try:
                                    realized_re  # Try this variable
                                except NameError:
                                    realized_re = 0 
                                

                                ############ Printing results #####################

                                if state.df_final['my_p'].iloc[-2] == 0:
                                    state.df_final['fee'] = abs(state.df_final['sig1']*state.df_final['trade_pr']*0.0001)
                                    state.df_final['pnl'] = -(state.df_final['sig1']*state.df_final['trade_pr']) - state.df_final['fee'] 

                                    #print('accumulated pnl', state.df_final['pnl'].fillna(0).sum())
                                    pct_ret = (state.df_final['pnl'].fillna(0).sum()/price1)*100
                                    state.realized_re = pct_ret
                                    print('accumulated % return', state.realized_re)
                                    state.ntrades = int(state.df_final['sig1'].fillna(0).abs().sum())
                                    print('number pf trade', state.ntrades)

                                else:

                                    print('realized % return', state.realized_re)
                                    # Extract current active position layer level and max allowed scale cap
                                    current_p_level = int(state.df_final['my_p'].iloc[-2])
                                    #max_pos = 3.0  # Equivalent to your max_pos normalization multiplier constant
                                    cl = float(state.df_final['close'].iloc[-1])

                                    # Isolate all historical entries where an actual trade purchase was executed (price > 0)
                                    valid_entries = state.df_final[state.df_final['trade_pr'] > 0]['trade_pr']

                                    if not valid_entries.empty:
                                        # CASE 1: Position is scaled up to level 2 (Average of the last 2 valid entries)
                                        if abs(current_p_level) == 2 and len(valid_entries) >= 2:
                                            # FIX: Extract the last two unique historical execution prices via safe .iloc positionals
                                            # This bypasses the empty intermediate zeros that corrupted your original math
                                            adj_price = (float(valid_entries.iloc[-1]) + float(valid_entries.iloc[-2])) / 2.0

                                            # Standardized trade return calculation applying the exposure direction vector sign
                                            trade_pnl_pct = ((cl / adj_price) - 1.0) * 100.0 * (current_p_level)
                                            print(f"[RISK N2] Adjusted Entry Price: {adj_price:.5f} | Scaled Trade P&L: {trade_pnl_pct:.4f}%", flush=True)

                                        # CASE 2: Position is scaled up to maximum exposure level 3 (Average of last 3 valid entries)
                                        elif abs(current_p_level) == 3 and len(valid_entries) >= 3:
                                            # FIX: Extract the last three unique execution milestones securely
                                            adj_price = (float(valid_entries.iloc[-1]) + float(valid_entries.iloc[-2]) + float(valid_entries.iloc[-3])) / 3.0

                                            trade_pnl_pct = ((cl / adj_price) - 1.0) * 100.0 * (current_p_level)
                                            print(f"[RISK N3] Adjusted Entry Price: {adj_price:.5f} | Maximum Trade P&L: {trade_pnl_pct:.4f}%", flush=True)

                                        # CASE 3: Standard single position layer execution baseline (Level 1)
                                        else:
                                            # Safely fetch the single last valid historical buy marker via dynamic lookup
                                            last_single_entry = float(valid_entries.iloc[-1])
                                            trade_pnl_pct = ((cl / last_single_entry) - 1.0) * 100.0 * (current_p_level)
                                            print(f"[RISK N1] Base Entry Price: {last_single_entry:.5f} | Standard Trade P&L: {trade_pnl_pct:.4f}%", flush=True)

                                        # Persistent state integration mapping for future decision logs
                                        state.df_final.loc[candle_time, 'trade_return_pct'] = trade_pnl_pct
                                    else:
                                        print("[RISK SYSTEM] No valid execution trade entries detected in historical log matrix.", flush=True)







                            except Exception as keras_error:
                                print(f"\n[KERAS INFERENCE CRASH] Error during neural network forwarding pass: {keras_error}", flush=True)  










                        # --- EXECUTE PREDICTION ---
                        # prediction = my_ml_model.predict(latest_sample_ready)
                        # print(f"[ML INFERENCE] Prediction Signal: {prediction}", flush=True)

                    else:
                        print("\n[ML INFERENCE WARNING] x_sig is empty after dropping NaNs. Skipping scaling.", flush=True)

                except Exception as scaling_error:
                    print(f"\n[ML INFERENCE CRASH] Error during scaling pipeline execution: {scaling_error}", flush=True)
            else:
                print("\n[ML INFERENCE WARNING] x_sig is missing or empty. Skipping scaling pipeline.", flush=True)



            # PRINT CONTROL: Displays cleanly exactly once every 60 seconds
            print(f"\n=============================================", flush=True)
            print(f"   [CANDLE CLOSED] MINUTE ENTRY #{state.count}   ", flush=True)
            print(f"=============================================", flush=True)
            print(f"[INFO] Global baseline reference price (price1): {price1:.5f}", flush=True)
            print("\nCumulative Trading Matrix (df_final):", flush=True)
            #print(state.df_final.tail(5), flush=True)

        except NameError as e:
            print(f"\n[PIPELINE WARNING] Missing technical indicator definition: {e}", flush=True)
            print("Please declare all required indicator functions before initiating the stream.", flush=True)
            return


        #print("[PIPELINE] Reached here successfully:", flush=True)
        print(df_tick, flush=True)

    except Exception as pipeline_error:
        import traceback
        print(f"[PIPELINE CRASH] Error: {pipeline_error}", flush=True)
        traceback.print_exc(file=sys.stdout)
        sys.stdout.flush()


if __name__ == "__main__":
    # SOLUÇÃO DEFINITIVA: Usando o domínio de fallback global da Binance para Futuros (fapi)
    # Esse endpoint alternativo não aplica o bloqueio silencioso que congela a thread anterior
    FALLBACK_URL = "wss://stream.binance.com/ws/ethusdt@kline_1m"
    #FALLBACK_URL = "wss://fstream.binance.com/ws/DOGEUSDT@kline_1m"

    print(f"Connecting to Binance Fallback Endpoint: {FALLBACK_URL}", flush=True)

    while True:
        try:
            # Conexão direta via Socket persistente
            ws = websocket.create_connection(
                FALLBACK_URL, 
                timeout=10,
                suppress_origin=True # Ignora restrição de cabeçalho do Jupyter/Colab
            )
            print("### Connection successfully open and authenticated! ###", flush=True)
            print("Streaming data now...", flush=True)

            while True:
                # O script para aqui e ESPERA o pacote chegar. Se o servidor enviar, o print roda no mesmo instante
                message = ws.recv()
                process_pipeline_sync(message)





        except websocket.WebSocketTimeoutException:
            print("[WARNING] Connection timeout. Re-pinging server...", flush=True)
            continue
        except KeyboardInterrupt:
            print("\nClosing WebSocket connection...", flush=True)
            if 'ws' in locals(): ws.close()
            print("Disconnected.", flush=True)
            break
        except Exception as e:
            print(f"### Connection lost ({e}). Reconnecting in 5 seconds... ###", flush=True)
            time.sleep(5)

Wiping old lingering WebSocket connection from memory...
Initializing clean WebSocket connection tunnel...
Executing emergency resource cleanup. Unlinking network sockets...
Success: System state restored. Ready for re-run without kernel restart!
### Production Scaler successfully loaded into memory! ###
Connecting to Binance Fallback Endpoint: wss://stream.binance.com/ws/ethusdt@kline_1m
### Connection successfully open and authenticated! ###
Streaming data now...

[CONFIRMED] CANDLE SAVED AT: 2026-11-14 15:17:00.015000 | Rows: 1

[WARM-UP] Accumulating history: 1/200 candles collected. Skipping math indicators...

[CONFIRMED] CANDLE SAVED AT: 2026-11-14 15:18:00.021000 | Rows: 2

[WARM-UP] Accumulating history: 2/200 candles collected. Skipping math indicators...

[CONFIRMED] CANDLE SAVED AT: 2026-11-14 15:19:00.023000 | Rows: 3

[WARM-UP] Accumulating history: 3/200 candles collected. Skipping math indicators...
